# PAI 最終課題：ことばで指示するとロボットが動く
日本語の指示 → LLM が行く順番を読みとる → MuJoCo のロボットが動く → 動画を書き出す

**最初に**：メニューの「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選ぶ

In [ ]:
!pip install -q mujoco imageio imageio-ffmpeg transformers accelerate

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'   # Colab で画面なしで描画するための設定

## robot_sim.py をアップロード
左の📁から `robot_sim.py` をドラッグしてもOK。下のセルを実行すると選ぶ画面が出るよ。

In [ ]:
from google.colab import files
import os
if not os.path.exists('robot_sim.py'):
    files.upload()

## AI（LLM）を読み込む
本番は Qwen2.5-3B-Instruct（下の実験の結果から選んだ）。推論だけで、学習はしない。

In [ ]:
from transformers import pipeline
llm = pipeline('text-generation', model='Qwen/Qwen2.5-3B-Instruct', device_map='auto')

def generate(prompt):
    msgs = [{'role': 'user', 'content': prompt}]
    out = llm(msgs, max_new_tokens=40, do_sample=False)
    return out[0]['generated_text'][-1]['content']

## 指示を出して動かす

In [ ]:
import robot_sim as sim
from IPython.display import Video

command = '青のボールに行ってから赤、最後に緑をタッチして'
order, how = sim.parse_command(command, generate)
print('指示:', command)
print('読みとった順番:', order, f'({how})')
path, sec = sim.run(order, 'robot.mp4')
print(f'{sec:.1f} 秒')
Video(path, embed=True)

In [ ]:
files.download('robot.mp4')

## 実験：モデルの大きさをくらべる
同じ5問で、0.5B / 1.5B / 3B / 7B が正しく順番を読みとれるかをくらべる。7B は T4 だとメモリが足りないので L4 以上で。
`robot_sim.py` のプロンプトにお手本（few-shot）を入れる前と後で、2回実行した。

In [ ]:
import gc, torch, importlib
from transformers import pipeline
import robot_sim as sim
importlib.reload(sim)

models = ["0.5B", "1.5B", "3B", "7B"]
tests = [
    ("青のボールに行ってから赤、最後に緑をタッチして", ["blue", "red", "green"]),
    ("赤に行って、そのあと青", ["red", "blue"]),
    ("最後に赤、その前に緑に行って", ["green", "red"]),
    ("緑と青に行って。青が先ね", ["blue", "green"]),
    ("赤、緑、青の順番で回って", ["red", "green", "blue"]),
]

for size in models:
    name = f"Qwen/Qwen2.5-{size}-Instruct"
    llm = pipeline("text-generation", model=name, device_map="auto", torch_dtype=torch.float16)
    def generate(prompt):
        out = llm([{"role": "user", "content": prompt}], max_new_tokens=40, do_sample=False)
        return out[0]["generated_text"][-1]["content"]

    ok = 0
    print(f"===== {size} =====")
    for cmd, answer in tests:
        order, how = sim.parse_command(cmd, generate)
        hit = order == answer
        ok += hit
        print("○" if hit else "×", cmd, "→", order)
    print(f"正解: {ok} / {len(tests)}\n")

    del llm; gc.collect(); torch.cuda.empty_cache()   # 次のモデルのためにメモリをあける

## 本番の動画をつくる
5つの指示を 3B に読ませて、タイトル → 各指示 → おわり の動画（約1分）にする。

In [ ]:
# 日本語フォントを入れる（動画に文字をのせるため）
!apt-get -qq install -y fonts-noto-cjk > /dev/null
FONT = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"

import importlib, robot_sim as sim
importlib.reload(sim)
from transformers import pipeline
import torch

llm = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct", device_map="auto", torch_dtype=torch.float16)
def generate(prompt):
    out = llm([{"role": "user", "content": prompt}], max_new_tokens=40, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

commands = [
    "青のボールに行ってから赤、最後に緑をタッチして",
    "赤に行って、そのあと青",
    "最後に赤、その前に緑に行って",
    "緑と青に行って。青が先ね",
    "赤、緑、青の順番で回って",
]
path, sec, results = sim.make_video(commands, generate, FONT, "final.mp4")
for cmd, order, how in results:
    print(cmd, "→", order, f"({how})")
print(f"{sec:.1f} 秒")

from IPython.display import Video
Video(path, embed=True)

In [ ]:
files.download('final.mp4')